# 08 — Register, reload, and use the approved artifact

**Plain-language question:** Which exact artifact receives the name `champion`,
and can a fresh loader reload it safely?

**Why this matters:** a release should point to the model that passed the gate,
not a convenient refit or an object still living only in one notebook kernel.

**Estimated time:** 45–60 minutes.
**Prerequisite:** lesson 07; you understand the fixed test decision and its
approved threshold.


## Preflight

Check the kernel and visibly locate or rebuild prerequisite evidence.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
import mlflow

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.inference import load_champion
from aai_local_classification.learning import state_exists
from aai_local_classification.modeling import feature_frame
from aai_local_classification.workflow import (
    get_or_run_candidate_selection,
    promote_if_approved,
    run_frozen_test_gate,
)

selection_missing = not state_exists("selection.json")
selection = get_or_run_candidate_selection(settings, root)
decision_missing = not state_exists("decision.json")
decision = run_frozen_test_gate(settings, root, selection)
print(
    f"Selection recreated: {selection_missing}; decision recreated: {decision_missing}"
)


### What you should see

After lessons 06–07, both values are `False`. If opened directly, this notebook
explicitly reports which prerequisite evidence it recreated.

### Words introduced

| Word | Plain meaning | Example |
|---|---|---|
| logged model | The saved model artifact attached to a run | selected Pipeline |
| registered version | An immutable numbered registry entry | version 1, 2, … |
| alias | A movable human-friendly pointer | `champion` |


## Follow the release chain

```text
candidate run
    └── logged Pipeline (fixed model ID / URI)
          └── frozen-test decision: ADOPT or REJECT
                └── registered version (only after ADOPT)
                      └── alias: champion
```

The concrete version is the auditable artifact. The alias helps consumers find
the current approved version and may move during a future release.


## Promote only an adopted model

**Before you run this:** predict what should happen if the gate says `reject`.
The safe behavior is a clear result with `registered=False`, not an exception
and not a moved alias.


In [ ]:
promotion = promote_if_approved(
    settings,
    decision,
    root,
    selection,
)
pd.Series(promotion).to_frame("promotion value")


### What you should see

For the deterministic course, `registered=True`, a concrete model version, and
alias `champion`. If a legitimate reject occurred, the table would instead
explain that the alias remains unchanged and the notebook would continue.

### How to interpret the output

Registration does not train anything. It records the exact selected model URI
that already passed the gate and adds release metadata around that artifact.


## Inspect the registry rather than trusting a success message

**Before you run this:** an alias is a pointer. Predict whether resolving it
should return the same concrete version shown in `promotion`.


In [ ]:
mlflow.set_tracking_uri(paths.tracking_uri)
client = mlflow.MlflowClient()

if promotion.get("registered"):
    registry_version = client.get_model_version_by_alias(
        settings.registered_model_name, "champion"
    )
    tags = registry_version.tags
    registry_view = pd.Series(
        {
            "registered_name": registry_version.name,
            "concrete_version": registry_version.version,
            "selected_candidate": tags.get("selected_candidate"),
            "decision_threshold": tags.get("decision_threshold"),
            "test_run_id": tags.get("test_run_id"),
        }
    )
else:
    registry_version = None
    registry_view = pd.Series({"status": promotion["reason"]})

registry_view.to_frame("registry value")


### What you should see

`champion` resolves to the same version number returned by promotion. Its tags
carry the selected candidate, threshold 0.12, and final test run ID. These fields
connect discovery (`champion`) back to release evidence.


## Inspect the input/output contract saved with the model

A **signature** describes expected input and output columns/types. An **input
example** is a small valid batch stored with the model to make that contract
concrete. Neither replaces live data validation.


In [ ]:
model_info = mlflow.models.get_model_info(selection.selected_model_uri)
signature_view = pd.Series(
    {
        "model_id": model_info.model_id,
        "input_schema": str(model_info.signature.inputs),
        "output_schema": str(model_info.signature.outputs),
        "decision_threshold_metadata": model_info.metadata.get("decision_threshold"),
        "positive_class_metadata": model_info.metadata.get("positive_class"),
    }
)
signature_view.to_frame("logged model contract")


### How to interpret the output

The nine raw feature names/types are the input contract. The model emits two
class probabilities. The approved binary action still requires the threshold
stored in registry evidence.

## Inspect the saved input example

The signature describes types; the input example shows five concrete rows that
match those types. It is documentation and a deployment check—not a substitute
for the complete training dataset or proof that all future rows will be valid.


In [ ]:
input_example_path = mlflow.artifacts.download_artifacts(
    artifact_uri=f"{model_info.artifact_path}/input_example.json"
)
saved_input_example = pd.read_json(input_example_path, orient="split")
saved_input_example


### What you should see

Five rows with the same nine feature columns named by the signature. The target
column is absent because inference does not require the future outcome.

### Misconception check

Moving an alias does not necessarily update a process that already loaded an
older model. Production jobs/endpoints should record the concrete version they
actually used.


## Reload into a new object and run inference

**Inference** means applying a fitted model to rows whose labels are not needed
at prediction time. We intentionally keep only declared feature columns.

**Before you run this:** predict the four output fields that a traceable
predictor should return.


In [ ]:
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
inference_batch = feature_frame(validation.tail(8), settings)

if promotion.get("registered"):
    predictor = load_champion(settings, root)
    prediction_output = predictor.predict(inference_batch, settings)
else:
    predictor = None
    prediction_output = pd.DataFrame(
        {"status": ["No approved champion; inference correctly skipped."]}
    )

prediction_output


### What you should see

Eight rows with `churn_probability`, `churn_prediction`, `model_name`, and the
concrete `model_version`. Probabilities are between 0 and 1, predictions are 0
or 1, and every row names the same loaded version.


In [ ]:
if predictor is not None:
    inference_preview = pd.concat(
        [
            inference_batch[["monthly_fee", "usage_hours_30d", "contract_type"]],
            prediction_output[["churn_probability", "churn_prediction"]],
        ],
        axis=1,
    )
else:
    inference_preview = prediction_output

inference_preview


### How to interpret the output

The feature values provide context for each score, but they do not establish a
causal explanation. The binary column is exactly
`churn_probability >= approved threshold`; it is not sklearn's default 0.5
classification.


In [ ]:
if predictor is not None:
    threshold_check = pd.DataFrame(
        {
            "probability": prediction_output.churn_probability,
            "approved_threshold": predictor.threshold,
            "recomputed_prediction": (
                prediction_output.churn_probability >= predictor.threshold
            ).astype(int),
            "returned_prediction": prediction_output.churn_prediction,
        }
    )
else:
    threshold_check = prediction_output

threshold_check


### Guided exercise

Classify a probability of 0.20 using thresholds 0.12 and 0.50. This isolates
why threshold evidence must travel with the model.


In [ ]:
exercise_probability = 0.20
exercise_actions = pd.Series(
    {
        "action_at_0.12": int(exercise_probability >= 0.12),
        "action_at_0.50": int(exercise_probability >= 0.50),
    }
)
exercise_actions.to_frame("binary action")


**Self-check:** the same model score becomes positive at 0.12 and negative at
0.50. Losing the approved threshold changes who receives the action.

<details><summary>Solution explanation</summary>

The comparisons are `0.20 >= 0.12` (true) and `0.20 >= 0.50` (false). The
threshold is release policy, not a hidden model default.
</details>


In [ ]:
# Reference solution — run after your attempt
assert exercise_actions.to_dict() == {
    "action_at_0.12": 1,
    "action_at_0.50": 0,
}
if predictor is not None:
    assert (
        threshold_check.recomputed_prediction == threshold_check.returned_prediction
    ).all()
print("✓ Version and approved threshold produce reproducible actions")


## MLOps bridge

Local SQLite Registry teaches the objects; Databricks uses Models in Unity
Catalog with a three-part name (`catalog.schema.model`) and governed
permissions. Use aliases for discovery and concrete versions for auditable jobs
or endpoints.

## Recap

- Only an adopted logged model is registered and assigned `champion`.
- A registered version is concrete evidence; an alias is a mutable pointer.
- Reloaded inference carries model name, version, and approved threshold.

**Evidence created:** a registered model version, version tags, `champion`
alias, and `promotion.json`. Compatible reruns resolve the same version rather
than registering duplicates.

**Ready for 09?** You can trace one prediction from alias to concrete version,
logged model, test decision, and threshold.
